# 2D Topology Optimization of a Cantilever Beam

This notebook sets up and solves a **minimum compliance** (strain energy) topology optimization problem using the classic **SIMP** material model and the **OC (Optimality Criteria)** update rule—i.e., the approach made popular by the “88-line” MATLAB code.

We’ll discretize a rectangular cantilever with bilinear quadrilateral (Q4) finite elements, fix the **left** edge, and apply a constant **downward point load** at the **midpoint of the right edge**. The design variables are element densities $x_e \in [x_{\min}, 1]$. We minimize compliance (maximize stiffness) subject to an overall volume fraction constraint. An **averaging filter** (a.k.a. sensitivity filter) is used to control mesh dependency and checkerboards.

The code that follows in a later cell will:

* assemble the FE system,
* perform SIMP-based compliance minimization with OC updates,
* and produce a lightweight animation (≤10 frames) showing the evolution of the density field from start to finish.

---

## Problem statement

Let the domain $\Omega \subset \mathbb{R}^2$ be meshed into $n$ Q4 elements indexed by $e=1,\dots,n$. For each element, the design variable $x_e \in [x_{\min},1]$ represents a relative density.

We seek

$$
\begin{aligned}
\min_{\mathbf{x}} \quad & c(\mathbf{x}) = \mathbf{f}^{\top}\mathbf{u}(\mathbf{x}) \\
\text{s.t.} \quad & \mathbf{K}(\mathbf{x})\mathbf{u}(\mathbf{x}) = \mathbf{f}, \\
& \frac{1}{n}\sum_{e=1}^n x_e \le \bar{V}, \\
& x_{\min} \le x_e \le 1,\quad e=1,\dots,n,
\end{aligned}
$$

where

* $c(\mathbf{x})$ is the compliance (equal to twice the strain energy in linear elasticity),
* $\mathbf{f}$ and ($\mathbf{u}$) are global load and displacement vectors,
* $\bar{V}$ is the prescribed **volume fraction**,
* $\mathbf{K}(\mathbf{x})$ is the global stiffness matrix assembled from element stiffnesses.

---

## Material interpolation (SIMP)

Each element stiffness is interpolated via the SIMP model:

$$
\mathbf{k}_e(x_e) = \big(E_{\min} + x_e^p(E_0 - E_{\min})\big)\mathbf{k}_0,
$$

where

* $E_0$ is the solid Young’s modulus (scaled to 1 without loss of generality),
* $E_{\min}$ is a small stiffness to avoid singularity (e.g. $10^{-9}E_0$),
* $p \ge 3$ is the **penalization** exponent,
* $\mathbf{k}_0$ is the reference element stiffness for $E=1$.

---

## Sensitivity analysis

Using the adjoint identity for compliance:

$$
\frac{\partial c}{\partial x_e}
= -\mathbf{u}^{\top}\frac{\partial \mathbf{K}}{\partial x_e}\mathbf{u}
= -px_e^{p-1}(E_0 - E_{\min})\mathbf{u}_e^{\top}\mathbf{k}_0\mathbf{u}_e,
$$

where $\mathbf{u}_e$ gathers the displacements of the element’s degrees of freedom.

We define the raw sensitivity $g_e := \frac{\partial c}{\partial x_e}$ (note $g_e \le 0$ for compliance).

---

## Sensitivity filtering (checkerboard control)

We use the **sensitivity filter** from the 88-line program: for each element $e$,

$$
\tilde{g}_e =
\frac{1}{\max(10^{-3}, x_e)}
\frac{\displaystyle\sum_{j \in \mathcal{N}(e)} H_{ej}x_jg_j}{\displaystyle\sum_{j \in \mathcal{N}(e)} H_{ej}},
\qquad
H_{ej} = \max\big(0, r_{\min} - | \mathbf{c}_e - \mathbf{c}_j | \big),
$$

where $\mathbf{c}*e$ is the centroid of element $e$, $r_{\min}$ is the filter radius in **element units**, and $\mathcal{N}(e)$ are elements within distance $r_{\min}$. This smooths sensitivities, discourages checkerboards, and introduces a minimum feature size on the order of $r*{\min}$.

---

## Design update: Optimality Criteria (OC) method

We solve the volume-constrained subproblem with an OC update and a bisection search for the Lagrange multiplier $\lambda$. With move limit $m \in (0,1]$ and current $x_e$, the trial update is

$$
x_e^{\text{trial}}(\lambda)
= \operatorname{clip} \left(
x_e \sqrt{\frac{-\tilde{g}_e}{\lambda}},
x_e - m, x_e + m
\right),
\quad \text{then} \quad
x_e^{\text{new}} = \operatorname{clip} \left(x_e^{\text{trial}}, x_{\min}, 1\right).
$$

We adjust $\lambda$ by bisection until the volume fraction constraint
$\frac{1}{n}\sum_e x_e^{\text{new}} = \bar{V}$ is met (within tolerance).

Default OC parameters:

* move limit $m=0.2$,
* $x_{\min}=10^{-3}$.

---

## Discretization and FE model

* Mesh: uniform $n_{\text{elx}} \times n_{\text{ely}}$ elements (columns × rows).
* Nodes: $(n_{\text{elx}}+1)\times(n_{\text{ely}}+1)$.
* DOFs: 2 per node $u, v$.
* Element: Q4 bilinear; we use the standard $2\times2$ Gauss rule and the closed-form $\mathbf{k}_0$ for $\nu=0.3$ (as in the 88-line code) or equivalently precompute $\mathbf{k}_0$ for given $\nu$ and unit $E$.
* Boundary conditions:

  * **Fixed** left edge: $u=v=0$ on all nodes with $x=0$.
  * **Load**: a unit **downward** force applied at the node located at the **mid-height of the right edge**.
    For even $n_{\text{ely}}$, “mid-height” is the lower of the two middle nodes; for odd $n_{\text{ely}}$, it’s the unique center node.

We assemble $\mathbf{K}$ from element contributions and enforce Dirichlet BCs in the usual way (remove fixed DOFs or apply large-penalty row/column modifications). The linear system
$\mathbf{K}\mathbf{u}=\mathbf{f}$ is symmetric positive definite after BCs.

> **Numerical note.** For portability in a notebook, we’ll use a robust solver path:
>
> * default: a lightweight Conjugate Gradient (CG) implementation on a sparse matrix (SciPy if available, else a simple NumPy CG on a matrix-vector multiply routine),
> * small default problem sizes ensure fast iterations if only NumPy is available.

---

## Stopping criteria

We iterate until either:

* the maximum change in densities falls below a tolerance,

  $$
  \Delta_\infty = \max_e |x_e^{k+1} - x_e^{k}| \le \varepsilon_{\text{change}}
  $$

* or a maximum number of iterations $k_{\max}$ is reached.

Typical choices: $\varepsilon_{\text{change}} = 10^{-3}$, $k_{\max} = 100\text{–}200$.

---

## Animation plan (≤10 frames)

To keep the output lightweight, we collect **at most 10 snapshots** of the density field ${x_e}$ that are **evenly spaced** over the full iteration history. For example, if $k_{\max}=120$ and we end at iteration $k^\star$, we choose indices

$$
\mathcal{K} = \left\{ \left \lfloor \frac{t}{N-1}k^\star \right \rfloor ~|~ t=0,\dots,N-1 \right\},\quad N \le 10,
$$

and render those frames as an in-notebook animation (HTML/JS) or a compact GIF.

---

## Default parameters (suggested)

* Grid: $n_{\text{elx}}=120, n_{\text{ely}}=40$ (you can reduce to $60\times20$ for faster runs).
* Volume fraction: $\bar{V}=0.5$.
* Penalization: $p=3.0$.
* Filter radius: $r_{\min}=1.5$ (in element units).
* Poisson’s ratio: $\nu=0.3$.
* $E_0=1,; E_{\min}=10^{-9}$.
* OC move limit $m=0.2$.
* Tolerances: $\varepsilon_{\text{change}}=10^{-3}$, $k_{\max}=150$.
* Animation frames: $N=10$ (or fewer if the algorithm converges much sooner).

---

## Algorithm outline

1. **Initialize** $x_e^{(0)} = \bar{V}$ for all elements.
2. **Repeat** for $k=0,1,2,\dots$ until convergence:

   1. **FE solve:** assemble $\mathbf{K}(\mathbf{x}^{(k)})$, apply BCs, solve $\mathbf{K}\mathbf{u}=\mathbf{f}$.
   2. **Compliance & sensitivities:** compute $c = \mathbf{f}^{\top}\mathbf{u}$ and raw $g_e = -p x_e^{p-1}(E_0 - E_{\min}),\mathbf{u}_e^{\top}\mathbf{k}_0\mathbf{u}_e$.
   3. **Filter sensitivities:** $\tilde{g} \leftarrow \text{Filter}(g,;x)$.
   4. **OC update:** find $\lambda$ by bisection so that the updated $x^{(k+1)}$ satisfies the volume fraction; apply move limits and box constraints.
   5. **Check convergence:** stop if $\Delta_\infty \le \varepsilon_{\text{change}}$ or $k=k_{\max}$.
   6. **(Optional)** Save frame if $k$ is in the selected animation indices.
3. **Display** final density field and the animation of saved frames.

---

## What you’ll see

* A density map (grayscale or colormap) showing material $(x\approx1)$ and void $(x\approx x_{\min})$.
* An animation (≤10 frames) that reveals the structure evolving from a uniform field to a familiar **cantilever beam** layout with tension/compression paths from the loaded tip to the fixed support.

---

In [8]:
# ===============================================================
# 2D Topology Optimization (SIMP + OC, 88-line-consistent) in Python
# Cantilever: left edge fully clamped (u=v=0), downward point load at right mid-height
# With animation overlays for BCs (left-edge fixed u & v) and the applied load.
# Redundant static animation frame suppressed.
# ===============================================================

import numpy as np
import math
import warnings
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# ----------------------------
# Problem parameters (editable)
# ----------------------------
nelx, nely = 120, 40          # elements: x-columns (nelx) × y-rows (nely)
volfrac      = 0.50           # target volume fraction
penal        = 3.0            # SIMP penalization exponent
rmin         = 1.5            # filter radius (in element units)
E0, Emin     = 1.0, 1e-9      # solid and minimum Young's modulus
nu           = 0.30           # Poisson's ratio (plane stress)
move_lim     = 0.20           # OC move limit per iteration
tol_change   = 1e-3           # convergence tolerance on max |Δx|
max_iter     = 150            # iteration cap
nframes_max  = 20             # #frames for the animation

# -----------------------------------------------------------------
# Optional SciPy sparse direct solver (faster). Fallback is NumPy CG
# -----------------------------------------------------------------
try:
    import scipy.sparse as sp
    import scipy.sparse.linalg as spla
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    warnings.warn(
        "SciPy not found; falling back to a NumPy-only Conjugate Gradient solver.\n"
        "For large meshes this can be slow. Consider reducing nelx/nely."
    )
    if nelx * nely > 2400:
        nelx, nely = 60, 20
        warnings.warn(
            f"Mesh automatically reduced to nelx={nelx}, nely={nely} for the NumPy-only solver."
        )

# ---------------------------------------------------
# 88-line style element stiffness matrix (plane stress)
# Ref: Sigmund's 88-line MATLAB code
# ---------------------------------------------------
def lk(nu: float) -> np.ndarray:
    k = np.array([
        1/2 - nu/6,   1/8 + nu/8,  -1/4 - nu/12, -1/8 + 3*nu/8,
       -1/4 + nu/12, -1/8 - nu/8,   nu/6,        1/8 - 3*nu/8
    ], dtype=float)
    KE = np.array([
        [ k[0], k[1], k[2], k[3], k[4], k[5], k[6], k[7]],
        [ k[1], k[0], k[7], k[6], k[5], k[4], k[3], k[2]],
        [ k[2], k[7], k[0], k[5], k[6], k[3], k[4], k[1]],
        [ k[3], k[6], k[5], k[0], k[1], k[2], k[7], k[4]],
        [ k[4], k[5], k[6], k[1], k[0], k[7], k[2], k[3]],
        [ k[5], k[4], k[3], k[2], k[7], k[0], k[1], k[6]],
        [ k[6], k[3], k[4], k[7], k[2], k[1], k[0], k[5]],
        [ k[7], k[2], k[1], k[4], k[3], k[6], k[5], k[0]]
    ], dtype=float)
    return KE

KE = lk(nu)
ke_flat = KE.ravel()

# -------------------------------------
# Node numbering & element DOF mapping
# (column-major like the 88-line code)
# -------------------------------------
# Node numbers in column-major order:
nodenrs = np.arange((nely + 1) * (nelx + 1), dtype=int).reshape((nely + 1, nelx + 1), order='F')

# Build per-element corner node arrays (no negative offsets)
nBL = nodenrs[:-1, :-1]  # bottom-left
nBR = nodenrs[:-1,  1:]  # bottom-right
nTR = nodenrs[ 1:,  1:]  # top-right
nTL = nodenrs[ 1:, :-1]  # top-left

# Stack to edofMat: [uBL, vBL, uBR, vBR, uTR, vTR, uTL, vTL]
edofMat = np.vstack((
    2*nBL.ravel(order='F'),
    2*nBL.ravel(order='F') + 1,
    2*nBR.ravel(order='F'),
    2*nBR.ravel(order='F') + 1,
    2*nTR.ravel(order='F'),
    2*nTR.ravel(order='F') + 1,
    2*nTL.ravel(order='F'),
    2*nTL.ravel(order='F') + 1
)).T

ndof = 2 * (nelx + 1) * (nely + 1)
nele = nelx * nely

# ------------------------------------------
# Boundary conditions (full clamp at x=0)
# ------------------------------------------
left_nodes = nodenrs[:, 0]                                  # all nodes on left edge
fixed_dofs = np.sort(np.r_[2*left_nodes, 2*left_nodes + 1]) # u=v=0 on left edge
all_dofs  = np.arange(ndof, dtype=int)
free_dofs = np.setdiff1d(all_dofs, fixed_dofs, assume_unique=True)

# ------------------------------------------
# Load: downward point at right mid-height
# ------------------------------------------
F = np.zeros(ndof)
mid_j = nely // 2
load_node = nodenrs[mid_j, -1]              # right edge, mid-height
F[2*load_node + 1] = -1.0                   # negative y-direction

# ------------------------------------------
# Precompute assembly structure for stiffness
# ------------------------------------------
iK = np.repeat(edofMat, 8, axis=1).ravel()
jK = np.tile(edofMat, (1, 8)).ravel()

# -----------------------------------------------------------
# Sensitivity filter (88-line style, distance weights)
# -----------------------------------------------------------
rmin_int = int(np.floor(rmin))
H_rows, H_cols, H_w = [], [], []
Hs = np.zeros(nele, dtype=float)

def elem_id(ix: int, jy: int) -> int:
    """Element index (column-major order)."""
    return jy + ix * nely

for ix in range(nelx):
    for jy in range(nely):
        e = elem_id(ix, jy)
        i0, i1 = max(ix - rmin_int, 0), min(ix + rmin_int, nelx - 1)
        j0, j1 = max(jy - rmin_int, 0), min(jy + rmin_int, nely - 1)
        for ii in range(i0, i1 + 1):
            for jj in range(j0, j1 + 1):
                dist = math.hypot(ix - ii, jy - jj)
                w = max(0.0, rmin - dist)
                if w > 0.0:
                    j = elem_id(ii, jj)
                    H_rows.append(e)
                    H_cols.append(j)
                    H_w.append(w)
                    Hs[e] += w

H_rows = np.array(H_rows, dtype=int)
H_cols = np.array(H_cols, dtype=int)
H_w    = np.array(H_w,    dtype=float)

# ----------------------------------
# OC update with sensitivity filtering
# ----------------------------------
def oc_update(x: np.ndarray,
              g: np.ndarray,
              move: float,
              volfrac: float,
              xmin: float = 1e-3) -> np.ndarray:
    """Filter raw sensitivities g and perform the OC bisection update."""
    nele = x.size

    # Sensitivity filter (88-line):
    numer = np.zeros(nele, dtype=float)
    np.add.at(numer, H_rows, H_w * x[H_cols] * g[H_cols])
    g_tilde = numer / (Hs * np.maximum(1e-3, x))

    # OC bisection on lambda
    l1, l2 = 0.0, 1e9
    x_new = x.copy()
    for _ in range(60):
        lam = 0.5 * (l1 + l2)
        ratio = np.maximum(0.0, -g_tilde / lam)
        xcand = x * np.sqrt(ratio)
        xcand = np.clip(xcand, x - move, x + move)   # move limit
        xcand = np.clip(xcand, xmin, 1.0)            # box constraints
        if xcand.mean() > volfrac:
            l1 = lam
        else:
            l2 = lam
        x_new = xcand
        if abs(x_new.mean() - volfrac) < 1e-5:
            break
    return x_new

# -----------------------------------------
# Linear solver wrappers (SciPy or NumPy CG)
# -----------------------------------------
def solve_system_scipy(sK: np.ndarray, F: np.ndarray, free: np.ndarray) -> np.ndarray:
    """Direct sparse solve using SciPy on the reduced system."""
    K = sp.coo_matrix((sK, (iK, jK)), shape=(ndof, ndof)).tocsr()
    K = 0.5 * (K + K.T)  # enforce symmetry numerically
    u = np.zeros(ndof)
    u[free] = spla.spsolve(K[free][:, free], F[free])
    return u

# NumPy-only reduced CG (if SciPy unavailable)
if not HAVE_SCIPY:
    map_full_to_free = -np.ones(ndof, dtype=int)
    map_full_to_free[free_dofs] = np.arange(free_dofs.size, dtype=int)
    mask_free = (map_full_to_free[iK] >= 0) & (map_full_to_free[jK] >= 0)
    iK_free_template = map_full_to_free[iK[mask_free]]
    jK_free_template = map_full_to_free[jK[mask_free]]

def cg_numpy_coo(i_idx: np.ndarray, j_idx: np.ndarray, val: np.ndarray,
                 b: np.ndarray, tol: float = 1e-8, maxiter: int = 4000) -> np.ndarray:
    """Minimal Conjugate Gradient for SPD matrices in COO triplets."""
    n = b.size
    x = np.zeros(n)
    def matvec(v):
        y = np.zeros_like(v)
        np.add.at(y, i_idx, val * v[j_idx])
        return y
    r = b - matvec(x)
    p = r.copy()
    rsold = float(np.dot(r, r))
    if rsold == 0.0:
        return x
    for _ in range(maxiter):
        Ap = matvec(p)
        pAp = float(np.dot(p, Ap))
        if pAp <= 1e-30:
            break
        alpha = rsold / pAp
        x += alpha * p
        r -= alpha * Ap
        rsnew = float(np.dot(r, r))
        if math.sqrt(rsnew) < tol:
            break
        p = r + (rsnew / rsold) * p
        rsold = rsnew
    return x

def solve_system_numpy_cg(sK: np.ndarray, F: np.ndarray, free: np.ndarray) -> np.ndarray:
    """Solve reduced system via NumPy CG using free-set triplets."""
    sK_free = sK[mask_free]
    u = np.zeros(ndof)
    u_free = cg_numpy_coo(iK_free_template, jK_free_template, sK_free, F[free], tol=1e-8)
    u[free] = u_free
    return u

# ----------------------------------------
# Main optimization loop + data collection
# ----------------------------------------
x = volfrac * np.ones(nele, dtype=float)   # start from uniform material
x_history = [x.copy()]
comp_history = []
change_history = []

for it in range(1, max_iter + 1):
    # SIMP interpolation
    E = Emin + (x ** penal) * (E0 - Emin)

    # Assemble global stiffness values (triplets share iK, jK)
    sK = np.tile(ke_flat, nele) * np.repeat(E, 64)

    # Solve K u = F
    if HAVE_SCIPY:
        u = solve_system_scipy(sK, F, free_dofs)
    else:
        u = solve_system_numpy_cg(sK, F, free_dofs)

    # Compliance & sensitivities
    ue = u[edofMat]                       # (nele, 8)
    ce = np.sum((ue @ KE) * ue, axis=1)   # element strain energy-like term
    compliance = float(F @ u)
    dc = -penal * (E0 - Emin) * (x ** (penal - 1.0)) * ce

    # OC update with sensitivity filtering
    x_new = oc_update(x, dc, move=move_lim, volfrac=volfrac, xmin=1e-3)

    # Convergence check
    change = float(np.max(np.abs(x_new - x)))
    x = x_new
    x_history.append(x.copy())
    comp_history.append(compliance)
    change_history.append(change)

    print(f"iter {it:3d} | c = {compliance:10.4f} | mean(x) = {x.mean():.4f} | change = {change:.4e}")
    if change < tol_change:
        print("Converged based on change tolerance.")
        break

# -------------------------------------------------------
# Visualization: compact (≤10) animation with BC/load overlay
# -------------------------------------------------------
# Select ≤10 evenly spaced frames over the entire history
n_total = len(x_history)
n_frames = min(nframes_max, n_total)
frame_ids = np.linspace(0, n_total - 1, num=n_frames, dtype=int)
frames = [x_history[k].reshape((nely, nelx), order='F') for k in frame_ids]  # match element ordering

# Animation (≤10 frames, to_jshtml for portability) with overlays
fig, ax = plt.subplots(figsize=(8, 3))

# Show density field; set extent so (x,y) are in element/node units
im = ax.imshow(frames[0], cmap='gray', interpolation='nearest', origin='lower',
               vmin=0, vmax=1, aspect='auto', extent=(0, nelx, 0, nely))
ttl = ax.set_title(f"Iteration {frame_ids[0]} / {n_total-1}")
ax.set_xlabel("Element index (x)")
ax.set_ylabel("Element index (y)")
plt.colorbar(im, ax=ax, label="density")

# ---- Overlay: boundary conditions (left edge fixed u & v for all nodes) ----
bc_len = 0.35  # length of tick marks
for y in range(nely + 1):
    # small horizontal tick (x-fix)
    ax.plot([0, min(bc_len, nelx)], [y, y], color='tab:blue', lw=1.0, zorder=5)
    # small vertical tick (y-fix)
    y2 = min(y + bc_len, nely)
    ax.plot([0, 0], [y, y2], color='tab:blue', lw=1.0, zorder=5)
ax.text(0.6, nely - 0.8, "fixed u,v", color='tab:blue', fontsize=9,
        ha='left', va='top', zorder=6)

# ---- Overlay: applied force (single downward at right mid-height) ----
fl = max(1.5, 0.06 * nely)  # visual arrow length
q = ax.quiver(nelx, mid_j, 0, -fl, angles='xy', scale_units='xy', scale=1,
              color='tab:red', width=0.006, zorder=6)
ax.text(nelx - 0.6, max(0.5, mid_j - fl - 0.5), "F", color='tab:red',
        fontsize=10, ha='right', va='top', zorder=6)

plt.tight_layout()

def init():
    im.set_data(frames[0])
    ttl.set_text(f"Iteration {frame_ids[0]} / {n_total-1}")
    return (im, ttl)

def animate(k):
    im.set_data(frames[k])
    ttl.set_text(f"Iteration {frame_ids[k]} / {n_total-1}")
    return (im, ttl)

ani = animation.FuncAnimation(fig, animate, init_func=init,
                              frames=len(frames), interval=600, blit=False, repeat=True)

# Produce HTML for the animation, then close the Matplotlib figure
# to avoid the redundant static frame at the end.
html_anim = HTML(ani.to_jshtml())
plt.close(fig)
html_anim


iter   1 | c =   980.6802 | mean(x) = 0.5000 | change = 2.0000e-01
iter   2 | c =   573.5261 | mean(x) = 0.5000 | change = 2.0000e-01
iter   3 | c =   404.3502 | mean(x) = 0.5000 | change = 2.0000e-01
iter   4 | c =   334.0604 | mean(x) = 0.5000 | change = 2.0000e-01
iter   5 | c =   311.3836 | mean(x) = 0.5000 | change = 2.0000e-01
iter   6 | c =   294.7094 | mean(x) = 0.5000 | change = 2.0000e-01
iter   7 | c =   282.1341 | mean(x) = 0.5000 | change = 2.0000e-01
iter   8 | c =   271.6617 | mean(x) = 0.5000 | change = 2.0000e-01
iter   9 | c =   264.4903 | mean(x) = 0.5000 | change = 1.7859e-01
iter  10 | c =   258.0908 | mean(x) = 0.5000 | change = 2.0000e-01
iter  11 | c =   252.8661 | mean(x) = 0.5000 | change = 1.8114e-01
iter  12 | c =   247.9001 | mean(x) = 0.5000 | change = 2.0000e-01
iter  13 | c =   243.4301 | mean(x) = 0.5000 | change = 1.7563e-01
iter  14 | c =   239.0569 | mean(x) = 0.5000 | change = 2.0000e-01
iter  15 | c =   233.9710 | mean(x) = 0.5000 | change = 2.0000